# ARC-v0.2 — Retriever-Condition Replication

## Main question

ARC-v0.1 found a potentially important phenomenon:

- iterative pseudo-relevance feedback often becomes stable without improving utility;
- wrong-attractor trajectories exist;
- oracle stopping has much more headroom than oracle improvement;
- retrieval-state features predict future trajectory changes.

But v0.1 used one ANN environment:

\[
\text{IVF-PQ}(M=32,\;8\text{ bit})
\]

Therefore the strongest alternative explanation is:

> Are wrong attractors merely an artifact of aggressive PQ approximation?

ARC-v0.2 directly tests that confound.

## Same queries, same feedback, three retrieval conditions

### C1 — IVF-PQ32
\[
32\text{ B/doc}
\]

The original ARC-v0.1 condition.

### C2 — IVF-PQ64
\[
64\text{ B/doc}
\]

Twice the PQ capacity.

### C3 — IVF-SQ8
Scalar-quantized 8-bit full-dimensional vectors:

\[
384\text{ B/doc}
\]

This retains far more per-dimension information and acts as a memory-safe
high-precision ANN condition.

If wrong-attractor behavior persists across C1 → C2 → C3, it becomes much harder to
attribute the phenomenon to PQ32 coding error alone.

## Feedback operators

The notebook automatically loads the latest ARC-v0.1 Stage-A grid and selects:

1. best `mean` feedback configuration;
2. best `softmax` feedback configuration;

using FIT-only oracle headroom.

No DEV or TEST query is searched.

## New prediction target

ARC-v0.1's `next_improves` target was extremely imbalanced.

v0.2 therefore studies both:

\[
P(\Delta U_{t+1}>0\mid S_t)
\]

and the more safety-relevant:

\[
\boxed{P(\Delta U_{t+1}<0\mid S_t)}
\]

We evaluate:
- within-condition group-cross-fitted ROC-AUC / AP;
- cross-condition transfer:
  train on IVF-PQ32 → test on IVF-PQ64 / IVF-SQ8.

This is a mechanism-replication experiment, not yet the final ARC controller.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
from datetime import datetime
import json, sys, subprocess, time, gc, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260816
DIM = 384

TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

NLIST = 4096
NPROBE = 64
NBITS = 8

TRAIN_DOCS = 200_000
ADD_BATCH = 10_000

REPLICATION_QUERY_COUNT = 1500

JACCARD_STABLE = 0.80
UTILITY_EPS = 1e-6
DRIFT_HIGH = 0.15

ROOT = Path(
    "/content/drive/MyDrive/"
    "hc-rars-fever-5m-untouched-confirmation-v1"
)

CORPUS_MEMMAP = ROOT / "stage1/corpus_embeddings.float16.memmap"
QUERY_EMB = ROOT / "stage1/query_embeddings_v2.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
FIT_QRELS = ROOT / "stage2/fit_qrels_rows.csv"

ARC_V0_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0"
)

CACHE_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

OUT = ARC_V0_ROOT / (
    "retriever-condition-replication-v02-"
    + datetime.now().strftime("%Y%m%d-%H%M%S")
)
OUT.mkdir(parents=True, exist_ok=True)

CONDITIONS = [
    {
        "name":"ivfpq32",
        "kind":"ivfpq",
        "M":32,
        "bytes_per_doc":32,
    },
    {
        "name":"ivfpq64",
        "kind":"ivfpq",
        "M":64,
        "bytes_per_doc":64,
    },
    {
        "name":"ivfsq8",
        "kind":"ivfsq8",
        "M":None,
        "bytes_per_doc":384,
    },
]

print("Output:", OUT)
print("Conditions:", CONDITIONS)


Output: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/retriever-condition-replication-v02-20260815-192204
Conditions: [{'name': 'ivfpq32', 'kind': 'ivfpq', 'M': 32, 'bytes_per_doc': 32}, {'name': 'ivfpq64', 'kind': 'ivfpq', 'M': 64, 'bytes_per_doc': 64}, {'name': 'ivfsq8', 'kind': 'ivfsq8', 'M': None, 'bytes_per_doc': 384}]


In [3]:
def run(cmd):
    print("$", " ".join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)), check=True)

run([
    sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu==1.12.0",
    "psutil",
    "pyarrow",
    "scikit-learn",
])

import faiss
import psutil

PROCESS = psutil.Process(os.getpid())

def ram_status(label=""):
    vm = psutil.virtual_memory()
    rss = PROCESS.memory_info().rss / 1024**3
    print(
        f"[RAM] {label:<30} "
        f"RSS={rss:6.2f} GB | "
        f"available={vm.available/1024**3:6.2f} GB | "
        f"used={vm.percent:5.1f}%"
    )

print("Faiss:", faiss.__version__)
print("RAM total GB:", psutil.virtual_memory().total / 1024**3)
ram_status("startup")


$ /usr/bin/python3 -m pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn
Faiss: 1.12.0
RAM total GB: 12.671417236328125
[RAM] startup                        RSS=  0.17 GB | available= 11.28 GB | used= 10.9%


## 1. Load FEVER artifacts — FIT only


In [4]:
size_bytes = CORPUS_MEMMAP.stat().st_size
bytes_per_row = DIM * np.dtype(np.float16).itemsize
assert size_bytes % bytes_per_row == 0

n_docs = size_bytes // bytes_per_row
assert n_docs == 5_416_568

docs = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(n_docs,DIM),
)

query_embeddings = np.load(
    QUERY_EMB,
    mmap_mode="r",
)
assert query_embeddings.shape == (123_142,DIM)

with open(QUERY_IDS,"r",encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

assert len(query_ids) == len(query_embeddings)
query_row = {qid:i for i,qid in enumerate(query_ids)}

with open(SPLIT_MANIFEST,"r",encoding="utf-8") as f:
    split = json.load(f)

fit_ids = [str(x).strip() for x in split["fit_query_ids"]]
dev_ids = [str(x).strip() for x in split["dev_query_ids"]]
test_ids = [str(x).strip() for x in split["test_query_ids"]]

assert split["test_retrieval_performed"] is False
assert split["test_relevance_values_accessed"] is False
assert all(q in query_row for q in fit_ids)

fit_rows = np.array(
    [query_row[q] for q in fit_ids],
    dtype=np.int64,
)

fit_qrels_df = pd.read_csv(FIT_QRELS)
fit_qrels_df["query-id"] = fit_qrels_df["query-id"].astype(str)

fit_qrels = {}
for qid,g in fit_qrels_df.groupby("query-id"):
    rel = g[g["score"]>0]["corpus-row"].astype(np.int64)
    fit_qrels[str(qid)] = set(rel.tolist())

print("Corpus:", docs.shape, docs.dtype)
print("FIT/DEV/TEST:", len(fit_ids),len(dev_ids),len(test_ids))
print("FIT qrels queries:", len(fit_qrels))
print("DEV/TEST retrieval in this notebook: prohibited")
ram_status("after data load")


Corpus: (5416568, 384) float16
FIT/DEV/TEST: 20000 6666 6666
FIT qrels queries: 20000
DEV/TEST retrieval in this notebook: prohibited
[RAM] after data load                RSS=  0.20 GB | available= 11.15 GB | used= 12.0%


## 2. Recover feedback configurations from ARC-v0.1


In [5]:
v01_runs = sorted(
    [
        p for p in ARC_V0_ROOT.glob("fever5m-observatory-v01-*")
        if (p/"stageA_grid_summary.csv").is_file()
    ],
    key=lambda p:p.stat().st_mtime,
)

if not v01_runs:
    raise FileNotFoundError(
        "No completed ARC-v0.1 run with stageA_grid_summary.csv found."
    )

V01_RUN = v01_runs[-1]
grid_summary = pd.read_csv(
    V01_RUN/"stageA_grid_summary.csv"
)

print("Using v0.1:", V01_RUN)
display(grid_summary.head(15))

selected_configs = []

for method in ["mean","softmax"]:
    t = grid_summary[
        grid_summary["method"] == method
    ].copy()

    if len(t):
        r = t.sort_values(
            ["oracle_minus_base","final_ndcg@10"],
            ascending=[False,False],
        ).iloc[0]

        selected_configs.append({
            "method":method,
            "k":int(r["k_feedback"]),
            "alpha":float(r["alpha"]),
            "temperature":(
                None
                if pd.isna(r["temperature"])
                else float(r["temperature"])
            ),
        })

if len(selected_configs) < 2:
    selected_configs = [
        {"method":"mean","k":10,"alpha":0.3,"temperature":None},
        {"method":"softmax","k":10,"alpha":0.3,"temperature":0.05},
    ]

print("Replication feedback configs:")
for c in selected_configs:
    print(c)


Using v0.1: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever5m-observatory-v01-20260815-185737


,method,k_feedback,alpha,temperature,base_ndcg@10,final_ndcg@10,oracle_ndcg@10,final_minus_base,oracle_minus_base,oracle_minus_final,harm_rate_final,improve_rate_final
0,softmax,5,0.5,0.10,0.14006,0.112756,0.142413,-0.027303,0.002353,0.029657,0.062,0.008
1,softmax,5,0.5,0.05,0.14006,0.121355,0.142387,-0.018705,0.002327,0.021033,0.046,0.008
2,softmax,20,0.3,0.10,0.14006,0.124124,0.142274,-0.015936,0.002214,0.018150,0.038,0.006
3,mean,20,0.3,NaN,0.14006,0.121798,0.142274,-0.018262,0.002214,0.020477,0.038,0.004
4,mean,10,0.5,NaN,0.14006,0.103474,0.142274,-0.036586,0.002214,0.038800,0.070,0.002
5,softmax,20,0.5,0.05,0.14006,0.102826,0.142274,-0.037234,0.002214,0.039448,0.072,0.004
6,softmax,20,0.5,0.10,0.14006,0.099191,0.142274,-0.040869,0.002214,0.043083,0.074,0.002
7,softmax,10,0.5,0.05,0.14006,0.109755,0.142167,-0.030304,0.002107,0.032412,0.062,0.006
8,softmax,10,0.3,0.10,0.14006,0.126565,0.141675,-0.013495,0.001615,0.015110,0.034,0.004
9,softmax,20,0.3,0.05,0.14006,0.124957,0.141675,-0.015103,0.001615,0.016718,0.036,0.004


Replication feedback configs:
{'method': 'mean', 'k': 20, 'alpha': 0.3, 'temperature': None}
{'method': 'softmax', 'k': 5, 'alpha': 0.5, 'temperature': 0.1}


## 3. Common replication query sample


In [6]:
rng = np.random.default_rng(SEED + 202)

replication_rows = rng.choice(
    fit_rows,
    size=REPLICATION_QUERY_COUNT,
    replace=False,
)

replication_qids = [
    query_ids[i] for i in replication_rows
]

print("Replication queries:", len(replication_rows))
print("First IDs:", replication_qids[:10])


Replication queries: 1500
First IDs: ['20238', '210791', '139823', '14315', '16021', '78037', '2896', '216936', '59271', '164330']


## 4. Persistent ANN index builders


In [7]:
def index_paths(condition):
    if condition["kind"] == "ivfpq":
        stem = (
            f"fever5m-bge-small-ivfpq-nlist{NLIST}-"
            f"m{condition['M']}-nbits{NBITS}-seed{SEED}"
        )
    else:
        stem = (
            f"fever5m-bge-small-ivfsq8-nlist{NLIST}-"
            f"seed{SEED}"
        )

    return (
        CACHE_ROOT/f"{stem}.faiss",
        CACHE_ROOT/f"{stem}.json",
    )

def expected_manifest(condition):
    return {
        "dataset":"FEVER",
        "corpus_rows":int(n_docs),
        "dimension":DIM,
        "metric":"inner_product",
        "nlist":NLIST,
        "seed":SEED,
        "train_docs":TRAIN_DOCS,
        "kind":condition["kind"],
        "M":condition["M"],
        "nbits":NBITS if condition["kind"]=="ivfpq" else 8,
    }

def manifest_matches(path,condition):
    if not path.is_file():
        return False
    try:
        old = json.loads(path.read_text(encoding="utf-8"))
        exp = expected_manifest(condition)
        return all(old.get(k)==v for k,v in exp.items())
    except Exception:
        return False

# One deterministic training sample shared by all retrievers.
train_ids = np.random.default_rng(SEED).choice(
    n_docs,
    size=min(TRAIN_DOCS,n_docs),
    replace=False,
)

def build_or_load_index(condition):
    idx_path, man_path = index_paths(condition)

    if idx_path.is_file() and manifest_matches(man_path,condition):
        print("Loading cached:", idx_path)
        index = faiss.read_index(str(idx_path))
        index.nprobe = NPROBE
        ram_status("after cached index load")
        return index

    print("Building:", condition)
    ram_status("before training vectors")

    train_x = np.ascontiguousarray(
        np.asarray(docs[train_ids],dtype=np.float32)
    )

    quantizer = faiss.IndexFlatIP(DIM)

    if condition["kind"] == "ivfpq":
        index = faiss.IndexIVFPQ(
            quantizer,
            DIM,
            NLIST,
            int(condition["M"]),
            NBITS,
            faiss.METRIC_INNER_PRODUCT,
        )
    elif condition["kind"] == "ivfsq8":
        index = faiss.IndexIVFScalarQuantizer(
            quantizer,
            DIM,
            NLIST,
            faiss.ScalarQuantizer.QT_8bit,
            faiss.METRIC_INNER_PRODUCT,
        )
    else:
        raise ValueError(condition)

    t0=time.time()
    print("Training...")
    index.train(train_x)
    print("Train minutes:",(time.time()-t0)/60)

    del train_x
    gc.collect()
    ram_status("after index train")

    print("Adding full corpus...")
    t0=time.time()

    for start in range(0,n_docs,ADD_BATCH):
        end=min(start+ADD_BATCH,n_docs)

        xb=np.asarray(
            docs[start:end],
            dtype=np.float32,
        )
        index.add(
            np.ascontiguousarray(xb)
        )
        del xb

        if end % 250_000 < ADD_BATCH or end == n_docs:
            gc.collect()
            print(
                f"{condition['name']}: "
                f"{end:,}/{n_docs:,} "
                f"({100*end/n_docs:.1f}%)"
            )
            ram_status(f"{condition['name']} add")

    print("Add minutes:",(time.time()-t0)/60)
    assert index.ntotal == n_docs

    index.nprobe = NPROBE

    print("Saving:",idx_path)
    faiss.write_index(index,str(idx_path))
    man_path.write_text(
        json.dumps(expected_manifest(condition),indent=2),
        encoding="utf-8",
    )

    ram_status("after index save")
    return index


## 5. Retrieval dynamics helpers


In [8]:
def normalize_rows(x):
    x=np.asarray(x,np.float32)
    n=np.linalg.norm(x,axis=1,keepdims=True)
    return x/np.maximum(n,1e-12)

def evaluate_one(qid,ranked_ids,k=10):
    relset=fit_qrels.get(str(qid),set())
    ranked=ranked_ids[:k]

    hits=np.array(
        [1.0 if int(d) in relset else 0.0 for d in ranked],
        dtype=np.float32,
    )

    recall=float(hits.sum()/max(len(relset),1))
    hitpos=np.flatnonzero(hits)
    mrr=float(1/(hitpos[0]+1)) if len(hitpos) else 0.0

    discounts=1.0/np.log2(np.arange(2,k+2))
    dcg=float((hits*discounts).sum())
    ideal=min(len(relset),k)
    idcg=float(discounts[:ideal].sum()) if ideal else 0.0
    ndcg=float(dcg/idcg) if idcg>0 else 0.0

    return recall,mrr,ndcg

def jaccard(a,b):
    a=set(map(int,a)); b=set(map(int,b))
    return len(a&b)/max(len(a|b),1)

def rank_overlap(a,b,k=10):
    return len(
        set(map(int,a[:k]))&
        set(map(int,b[:k]))
    )/k

def score_entropy(scores,temperature=.05):
    z=np.asarray(scores,np.float64)/temperature
    z-=z.max()
    p=np.exp(np.clip(z,-60,60))
    p/=np.maximum(p.sum(),1e-12)
    return float(-np.sum(p*np.log(np.maximum(p,1e-12))))

def candidate_dispersion(ids):
    x=normalize_rows(
        np.asarray(
            docs[np.asarray(ids[:20],np.int64)],
            dtype=np.float32,
        )
    )
    centroid=normalize_rows(
        x.mean(axis=0,keepdims=True)
    )[0]
    return float(np.mean(1-x@centroid))

def state_features(q0,qt,scores,ids,prev_ids):
    anchor=float(
        np.dot(q0,qt)/
        max(np.linalg.norm(q0)*np.linalg.norm(qt),1e-12)
    )

    out={
        "query_drift":1-anchor,
        "score_entropy":score_entropy(scores[:20]),
        "score_margin_1_2":float(scores[0]-scores[1]),
        "score_margin_10_11":float(scores[9]-scores[10]),
        "candidate_dispersion":candidate_dispersion(ids),
        "candidate_jaccard":np.nan,
        "rank_overlap@10":np.nan,
    }

    if prev_ids is not None:
        out["candidate_jaccard"]=jaccard(
            ids[:TOP_RETRIEVE],
            prev_ids[:TOP_RETRIEVE],
        )
        out["rank_overlap@10"]=rank_overlap(
            ids,prev_ids,TOP_K
        )

    return out

def feedback_vector(ids,scores,config):
    k=config["k"]
    dids=np.asarray(ids[:k],np.int64)
    x=np.asarray(docs[dids],dtype=np.float32)

    if config["method"]=="mean":
        f=x.mean(axis=0)
    else:
        z=np.asarray(scores[:k],np.float64)/float(config["temperature"])
        z-=z.max()
        w=np.exp(np.clip(z,-60,60))
        w/=np.maximum(w.sum(),1e-12)
        f=(x*w[:,None]).sum(axis=0)

    f=f.astype(np.float32)
    return f/max(np.linalg.norm(f),1e-12)

def anchored_update(q0,f,alpha):
    q=((1-alpha)*q0+alpha*f).astype(np.float32)
    return q/max(np.linalg.norm(q),1e-12)


In [9]:
def run_condition_trajectory(index,condition,config):
    qrows=replication_rows
    qids=replication_qids

    q0=normalize_rows(
        np.asarray(
            query_embeddings[qrows],
            dtype=np.float32,
        )
    )
    qt=q0.copy()

    prev_ids=[None]*len(qrows)
    rows=[]

    for t in range(MAX_ROUNDS+1):
        print(
            condition["name"],
            config["method"],
            "iteration",t,
        )

        scores,ids=index.search(
            np.ascontiguousarray(qt,np.float32),
            TOP_RETRIEVE,
        )

        next_q=np.empty_like(qt)

        for i in range(len(qrows)):
            recall,mrr,ndcg=evaluate_one(
                qids[i],ids[i],TOP_K
            )

            sf=state_features(
                q0[i],qt[i],scores[i],ids[i],prev_ids[i]
            )

            rows.append({
                "condition":condition["name"],
                "bytes_per_doc":condition["bytes_per_doc"],
                "query_id":qids[i],
                "query_row":int(qrows[i]),
                "method":config["method"],
                "k_feedback":config["k"],
                "alpha":config["alpha"],
                "temperature":(
                    np.nan if config["temperature"] is None
                    else config["temperature"]
                ),
                "iteration":t,
                "recall@10":recall,
                "mrr@10":mrr,
                "ndcg@10":ndcg,
                **sf,
            })

            if t<MAX_ROUNDS:
                f=feedback_vector(
                    ids[i],scores[i],config
                )
                next_q[i]=anchored_update(
                    q0[i],f,config["alpha"]
                )

            prev_ids[i]=ids[i].copy()

        if t<MAX_ROUNDS:
            qt=next_q

        ram_status(f"{condition['name']} t={t}")

    return pd.DataFrame(rows)


## 6. Run conditions sequentially

Each index is loaded/built, evaluated, saved, and released before the next condition.
This avoids keeping multiple 5.4M-document indices in RAM simultaneously.


In [10]:
trajectory_paths=[]

for condition in CONDITIONS:
    print("="*100)
    print("CONDITION:",condition)

    index=build_or_load_index(condition)
    index.nprobe=NPROBE

    condition_frames=[]

    for cfg in selected_configs:
        df=run_condition_trajectory(
            index,
            condition,
            cfg,
        )
        condition_frames.append(df)

    condition_df=pd.concat(
        condition_frames,
        ignore_index=True,
    )

    path=OUT/f"{condition['name']}_trajectories.parquet"
    condition_df.to_parquet(path,index=False)
    trajectory_paths.append(path)

    print("Saved:",path)

    del condition_df,index
    gc.collect()
    ram_status(f"released {condition['name']}")

print("All conditions completed.")


CONDITION: {'name': 'ivfpq32', 'kind': 'ivfpq', 'M': 32, 'bytes_per_doc': 32}
Building: {'name': 'ivfpq32', 'kind': 'ivfpq', 'M': 32, 'bytes_per_doc': 32}
[RAM] before training vectors        RSS=  0.20 GB | available= 11.57 GB | used=  8.7%
Training...
Train minutes: 2.399107336997986
[RAM] after index train              RSS=  3.75 GB | available= 10.84 GB | used= 14.5%
Adding full corpus...
ivfpq32: 250,000/5,416,568 (4.6%)
[RAM] ivfpq32 add                    RSS=  3.81 GB | available= 10.79 GB | used= 14.8%
ivfpq32: 500,000/5,416,568 (9.2%)
[RAM] ivfpq32 add                    RSS=  3.85 GB | available= 10.77 GB | used= 15.0%
ivfpq32: 750,000/5,416,568 (13.8%)
[RAM] ivfpq32 add                    RSS=  3.87 GB | available= 10.78 GB | used= 14.9%
ivfpq32: 1,000,000/5,416,568 (18.5%)
[RAM] ivfpq32 add                    RSS=  3.89 GB | available= 10.80 GB | used= 14.7%
ivfpq32: 1,250,000/5,416,568 (23.1%)
[RAM] ivfpq32 add                    RSS=  3.92 GB | available= 10.79 GB | used

## 7. Load trajectories and assign attractor labels


In [11]:
all_df=pd.concat(
    [pd.read_parquet(p) for p in trajectory_paths],
    ignore_index=True,
)

group_cols=[
    "condition",
    "query_id",
    "method",
    "k_feedback",
    "alpha",
    "temperature",
]

label_rows=[]
oracle_rows=[]
step_rows=[]

for key,g in all_df.groupby(group_cols,dropna=False):
    g=g.sort_values("iteration").reset_index(drop=True)

    base=float(g.iloc[0]["ndcg@10"])
    final=float(g.iloc[-1]["ndcg@10"])
    final_j=float(g.iloc[-1]["candidate_jaccard"])
    drift=float(g.iloc[-1]["query_drift"])
    delta=final-base

    if not np.isnan(final_j) and final_j>=JACCARD_STABLE:
        if delta>UTILITY_EPS:
            label="correct_convergence"
        elif delta<-UTILITY_EPS:
            label="wrong_attractor"
        else:
            label="saturation"
    elif delta<-UTILITY_EPS and drift>=DRIFT_HIGH:
        label="drift"
    elif delta>UTILITY_EPS:
        label="productive_exploration"
    else:
        label="unstable_no_gain"

    utilities=g["ndcg@10"].to_numpy()
    bestpos=int(np.argmax(utilities))

    label_rows.append({
        "condition":key[0],
        "query_id":key[1],
        "method":key[2],
        "base_ndcg":base,
        "final_ndcg":final,
        "delta_ndcg":delta,
        "final_jaccard":final_j,
        "final_query_drift":drift,
        "trajectory_label":label,
    })

    oracle_rows.append({
        "condition":key[0],
        "query_id":key[1],
        "method":key[2],
        "base_ndcg":base,
        "final_ndcg":final,
        "oracle_ndcg":float(utilities[bestpos]),
        "oracle_iteration":int(g.iloc[bestpos]["iteration"]),
        "oracle_minus_base":float(utilities[bestpos]-base),
        "oracle_minus_final":float(utilities[bestpos]-final),
    })

    for i in range(len(g)-1):
        cur=g.iloc[i]
        nxt=g.iloc[i+1]

        delta_next=float(
            nxt["ndcg@10"]-cur["ndcg@10"]
        )

        step_rows.append({
            "condition":key[0],
            "query_id":key[1],
            "method":key[2],
            "iteration":int(cur["iteration"]),
            "query_drift":float(cur["query_drift"]),
            "candidate_jaccard":(
                0.0 if pd.isna(cur["candidate_jaccard"])
                else float(cur["candidate_jaccard"])
            ),
            "rank_overlap@10":(
                0.0 if pd.isna(cur["rank_overlap@10"])
                else float(cur["rank_overlap@10"])
            ),
            "score_entropy":float(cur["score_entropy"]),
            "score_margin_1_2":float(cur["score_margin_1_2"]),
            "score_margin_10_11":float(cur["score_margin_10_11"]),
            "candidate_dispersion":float(cur["candidate_dispersion"]),
            "delta_next_ndcg":delta_next,
            "next_improves":int(delta_next>UTILITY_EPS),
            "next_harms":int(delta_next<-UTILITY_EPS),
        })

labels_df=pd.DataFrame(label_rows)
oracle_df=pd.DataFrame(oracle_rows)
step_df=pd.DataFrame(step_rows)

display(
    pd.crosstab(
        labels_df["condition"],
        labels_df["trajectory_label"],
        normalize="index",
    )
)


trajectory_label,correct_convergence,productive_exploration,saturation,unstable_no_gain,wrong_attractor
condition,,,,,
ivfpq32,0.008000,0.000333,0.900000,0.030,0.061667
ivfpq64,0.007667,0.000333,0.921667,0.039,0.031333
ivfsq8,0.011000,0.000000,0.937000,0.036,0.016000


## 8. Replication summary


In [12]:
summary_rows=[]

for condition in [c["name"] for c in CONDITIONS]:
    lab=labels_df[labels_df.condition==condition]
    ora=oracle_df[oracle_df.condition==condition]

    summary_rows.append({
        "condition":condition,
        "wrong_attractor_rate":float(
            np.mean(lab.trajectory_label=="wrong_attractor")
        ),
        "correct_convergence_rate":float(
            np.mean(lab.trajectory_label=="correct_convergence")
        ),
        "saturation_rate":float(
            np.mean(lab.trajectory_label=="saturation")
        ),
        "drift_rate":float(
            np.mean(lab.trajectory_label=="drift")
        ),
        "mean_final_minus_base_ndcg":float(
            (ora.final_ndcg-ora.base_ndcg).mean()
        ),
        "mean_oracle_minus_base_ndcg":float(
            ora.oracle_minus_base.mean()
        ),
        "mean_oracle_stopping_headroom":float(
            ora.oracle_minus_final.mean()
        ),
    })

summary_df=pd.DataFrame(summary_rows)
display(summary_df)


,condition,wrong_attractor_rate,correct_convergence_rate,saturation_rate,drift_rate,mean_final_minus_base_ndcg,mean_oracle_minus_base_ndcg,mean_oracle_stopping_headroom
0,ivfpq32,0.061667,0.008000,0.900000,0.0,-0.025517,0.002260,0.027777
1,ivfpq64,0.031333,0.007667,0.921667,0.0,-0.008695,0.002749,0.011445
2,ivfsq8,0.016000,0.011000,0.937000,0.0,-0.001681,0.002893,0.004574


## 9. Within-condition harm / improvement predictability


In [13]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols=[
    "iteration",
    "query_drift",
    "candidate_jaccard",
    "rank_overlap@10",
    "score_entropy",
    "score_margin_1_2",
    "score_margin_10_11",
    "candidate_dispersion",
]

def make_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED,
        ),
    )

prediction_rows=[]

for condition in [c["name"] for c in CONDITIONS]:
    sdf=step_df[step_df.condition==condition].copy()

    X=sdf[feature_cols].to_numpy(np.float32)
    groups=sdf.query_id.astype(str).to_numpy()

    for target in ["next_harms","next_improves"]:
        y=sdf[target].to_numpy(np.int32)

        if len(np.unique(y))<2:
            prediction_rows.append({
                "condition":condition,
                "target":target,
                "positive_rate":float(y.mean()),
                "roc_auc":np.nan,
                "average_precision":np.nan,
            })
            continue

        pred=cross_val_predict(
            make_model(),
            X,y,
            groups=groups,
            cv=GroupKFold(n_splits=5),
            method="predict_proba",
        )[:,1]

        prediction_rows.append({
            "condition":condition,
            "target":target,
            "positive_rate":float(y.mean()),
            "roc_auc":float(roc_auc_score(y,pred)),
            "average_precision":float(
                average_precision_score(y,pred)
            ),
        })

prediction_df=pd.DataFrame(prediction_rows)
display(prediction_df)


,condition,target,positive_rate,roc_auc,average_precision
0,ivfpq32,next_harms,0.025083,0.725926,0.074496
1,ivfpq32,next_improves,0.004167,0.805800,0.021950
2,ivfpq64,next_harms,0.012500,0.739039,0.035749
3,ivfpq64,next_improves,0.003250,0.759372,0.027208
4,ivfsq8,next_harms,0.006083,0.722380,0.019598
5,ivfsq8,next_improves,0.003833,0.763238,0.040215


## 10. Cross-retriever transfer

Train a safety model only on IVF-PQ32 states and labels, then evaluate it without retraining
on IVF-PQ64 and IVF-SQ8.

A transferable harm signal is much stronger evidence than an in-condition classifier.


In [14]:
source=step_df[
    step_df.condition=="ivfpq32"
].copy()

Xsrc=source[feature_cols].to_numpy(np.float32)
ysrc=source["next_harms"].to_numpy(np.int32)

transfer_model=make_model()
transfer_model.fit(Xsrc,ysrc)

transfer_rows=[]

for target_condition in ["ivfpq64","ivfsq8"]:
    tdf=step_df[
        step_df.condition==target_condition
    ].copy()

    X=tdf[feature_cols].to_numpy(np.float32)
    y=tdf["next_harms"].to_numpy(np.int32)

    p=transfer_model.predict_proba(X)[:,1]

    transfer_rows.append({
        "train_condition":"ivfpq32",
        "test_condition":target_condition,
        "harm_rate":float(y.mean()),
        "roc_auc":(
            float(roc_auc_score(y,p))
            if len(np.unique(y))>1 else np.nan
        ),
        "average_precision":(
            float(average_precision_score(y,p))
            if len(np.unique(y))>1 else np.nan
        ),
    })

transfer_df=pd.DataFrame(transfer_rows)
display(transfer_df)


,train_condition,test_condition,harm_rate,roc_auc,average_precision
0,ivfpq32,ivfpq64,0.012500,0.748522,0.030257
1,ivfpq32,ivfsq8,0.006083,0.700133,0.011441


## 11. Effect-size comparison across retrievers


In [15]:
pivot=labels_df.pivot_table(
    index=["query_id","method"],
    columns="condition",
    values="delta_ndcg",
    aggfunc="first",
)

display(pivot.describe())

if {"ivfpq32","ivfsq8"}.issubset(pivot.columns):
    valid=pivot[["ivfpq32","ivfsq8"]].dropna()

    same_harm=np.mean(
        (valid["ivfpq32"]<0)
        & (valid["ivfsq8"]<0)
    )

    sign_agree=np.mean(
        np.sign(valid["ivfpq32"])
        == np.sign(valid["ivfsq8"])
    )

    corr=float(
        np.corrcoef(
            valid["ivfpq32"],
            valid["ivfsq8"],
        )[0,1]
    )

    print("PQ32 & SQ8 both final-harm fraction:",same_harm)
    print("PQ32/SQ8 delta-sign agreement       :",sign_agree)
    print("PQ32/SQ8 delta correlation          :",corr)


condition,ivfpq32,ivfpq64,ivfsq8
count,3000.000000,3000.000000,3000.000000
mean,-0.025517,-0.008695,-0.001681
std,0.125507,0.082963,0.057400
min,-1.000000,-1.000000,-1.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,0.369070,1.000000,1.000000


PQ32 & SQ8 both final-harm fraction: 0.006333333333333333
PQ32/SQ8 delta-sign agreement       : 0.921
PQ32/SQ8 delta correlation          : 0.08881374942715999


## 12. ARC-v0.2 replication decision


In [16]:
S=summary_df.set_index("condition")

pq32_wrong=float(S.loc["ivfpq32","wrong_attractor_rate"])
pq64_wrong=float(S.loc["ivfpq64","wrong_attractor_rate"])
sq8_wrong=float(S.loc["ivfsq8","wrong_attractor_rate"])

sq8_stop=float(
    S.loc["ivfsq8","mean_oracle_stopping_headroom"]
)

transfer_auc_mean=float(
    transfer_df["roc_auc"].mean()
)

print("=== ARC-v0.2 RETRIEVER-CONDITION REPLICATION ===")
print(f"Wrong attractor — IVF-PQ32 : {pq32_wrong:.3%}")
print(f"Wrong attractor — IVF-PQ64 : {pq64_wrong:.3%}")
print(f"Wrong attractor — IVF-SQ8  : {sq8_wrong:.3%}")
print(f"SQ8 oracle stopping headroom: {sq8_stop:+.6f}")
print(f"PQ32→other harm AUC mean    : {transfer_auc_mean:.4f}")
print()

persists_high_precision = sq8_wrong >= 0.02
persists_all = min(pq32_wrong,pq64_wrong,sq8_wrong) >= 0.02
stopping_real = sq8_stop >= 0.01
transferable = transfer_auc_mean >= 0.65

if persists_all and stopping_real and transferable:
    decision=(
        "STRONG RETRIEVER-INVARIANT GO: wrong-attractor behavior persists from PQ32 through "
        "higher-capacity PQ64 and SQ8, oracle stopping remains valuable, and harm-state signals "
        "transfer across retrievers. Proceed to ARC-v0.3 cross-dataset replication."
    )
elif persists_high_precision and stopping_real:
    decision=(
        "PHENOMENON REPLICATES, SIGNAL PARTLY RETRIEVER-SPECIFIC: wrong attractors persist in "
        "the high-precision SQ8 condition, so PQ32 approximation alone cannot explain them. "
        "Proceed to cross-dataset replication, but improve state normalization before controller work."
    )
elif not persists_high_precision:
    decision=(
        "PQ-ARTIFACT WARNING: wrong-attractor prevalence collapses in SQ8. The current phenomenon "
        "may be driven substantially by compressed retrieval error; do not claim general iterative "
        "retrieval dynamics without a new formulation."
    )
else:
    decision=(
        "MIXED REPLICATION: the phenomenon is present but not stable enough across retriever "
        "conditions for a strong claim. Inspect per-feedback and per-query paired effects."
    )

print("DECISION:",decision)


=== ARC-v0.2 RETRIEVER-CONDITION REPLICATION ===
Wrong attractor — IVF-PQ32 : 6.167%
Wrong attractor — IVF-PQ64 : 3.133%
Wrong attractor — IVF-SQ8  : 1.600%
SQ8 oracle stopping headroom: +0.004574
PQ32→other harm AUC mean    : 0.7243

DECISION: PQ-ARTIFACT WARNING: wrong-attractor prevalence collapses in SQ8. The current phenomenon may be driven substantially by compressed retrieval error; do not claim general iterative retrieval dynamics without a new formulation.


## 13. Save all evidence


In [17]:
labels_df.to_csv(
    OUT/"retriever_condition_labels.csv",
    index=False,
)
oracle_df.to_csv(
    OUT/"retriever_condition_oracle.csv",
    index=False,
)
step_df.to_parquet(
    OUT/"retriever_condition_steps.parquet",
    index=False,
)
summary_df.to_csv(
    OUT/"retriever_condition_summary.csv",
    index=False,
)
prediction_df.to_csv(
    OUT/"within_condition_prediction.csv",
    index=False,
)
transfer_df.to_csv(
    OUT/"cross_condition_transfer.csv",
    index=False,
)

report={
    "design":{
        "dataset":"FEVER",
        "split":"FIT only",
        "queries":REPLICATION_QUERY_COUNT,
        "rounds":MAX_ROUNDS,
        "nlist":NLIST,
        "nprobe":NPROBE,
        "conditions":CONDITIONS,
        "feedback_configs":selected_configs,
        "test_retrieval_performed":False,
    },
    "source_v01":str(V01_RUN),
    "summary":summary_df.to_dict(orient="records"),
    "within_condition_prediction":prediction_df.to_dict(orient="records"),
    "cross_condition_transfer":transfer_df.to_dict(orient="records"),
    "decision":decision,
}

(OUT/"report.json").write_text(
    json.dumps(report,indent=2,default=float),
    encoding="utf-8",
)

print("Saved:",OUT)
print("Report:",OUT/"report.json")


Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/retriever-condition-replication-v02-20260815-192204
Report: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/retriever-condition-replication-v02-20260815-192204/report.json
